# Validador de Procesos contra Arquetipos y Mapa de Capacidades

Este notebook carga un archivo de actividades de procesos, lo cruza contra el catálogo de arquetipos para identificar a cuál arquetipo pertenece cada código de proceso, y luego busca coincidencias entre las actividades y el mapa de capacidades de procesos.

## 1. Importación de librerías

Se importa `pandas`, la librería principal para manipulación de datos tabulares en este notebook.

In [2]:
import pandas as pd


## 2. Carga del archivo de actividades

Lee el archivo `data/query_actividades_proc_q_actual.csv` y lo carga en el DataFrame `df`.

In [3]:
from pathlib import Path
import pandas as pd

csv_path = Path("data/query_actividades_proc_q_actual.csv")
df = pd.read_csv(csv_path, dtype=str)

print(f"Archivo cargado: {csv_path}")
print(f"{len(df)} filas · {len(df.columns)} columnas")
print(f"Columnas: {list(df.columns)}")
df.head()

Archivo cargado: data\query_actividades_proc_q_actual.csv
39033 filas · 5 columnas
Columnas: ['t1.id_epica', 't1.periodo', 'codigo_proceso', 't2.procedimiento', 't2.actividad']


,t1.id_epica,t1.periodo,codigo_proceso,t2.procedimiento,t2.actividad
0,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,bloqueo usuario
1,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,solicitar impugnación
2,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,solicitar autorización
3,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,s: crear o vincular clientes
4,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,revisión inconsistencias


## 3. Códigos de proceso únicos × Arquetipos (tabla expandida)

Extrae los `codigo_proceso` únicos del CSV de actividades y los cruza contra `arquetipos.csv`.  
El resultado tiene **una fila por cada par (codigo_proceso, arquetipo)**: un proceso puede aparecer en varios arquetipos y un arquetipo puede contener varios procesos.

In [4]:
import sys, os, importlib
sys.path.insert(0, os.getcwd())

import helper.match_arquetipos as _ma
importlib.reload(_ma)
from helper.match_arquetipos import load_arquetipos

# ── 1. Códigos de proceso únicos ─────────────────────────────────────────────
df_codigos = (
    df[['codigo_proceso']]
    .drop_duplicates()
    .assign(codigo_proceso=lambda x: x['codigo_proceso'].astype(str).str.strip())
    .reset_index(drop=True)
)

# ── 2. Arquetipos explodidos (una fila por código de proceso en el catálogo) ──
_, df_arq_expl = load_arquetipos('data/arquetipos.csv')

arq_code_col = next(c for c in df_arq_expl.columns if 'Código' in c and 'Arquetipo' in c)
arq_name_col = next(c for c in df_arq_expl.columns if 'Nombre' in c and 'Arquetipo' in c)

# ── 3. Left join: conserva todos los códigos, N/A si no hay arquetipo ────────
df_proc_arquetipos = (
    df_codigos
    .merge(
        df_arq_expl[['proc_code', arq_code_col, arq_name_col]].drop_duplicates(),
        left_on='codigo_proceso',
        right_on='proc_code',
        how='left',
    )
    [['codigo_proceso', arq_code_col, arq_name_col]]
    .fillna('N/A')
    .sort_values(['codigo_proceso', arq_code_col])
    .reset_index(drop=True)
)

sin_arquetipo = (df_proc_arquetipos[arq_code_col] == 'N/A').sum()
print(f"Códigos únicos en el CSV:            {df['codigo_proceso'].nunique()}")
print(f"Con al menos un arquetipo asociado:  {df['codigo_proceso'].nunique() - sin_arquetipo}")
print(f"Sin ningún arquetipo (N/A):          {sin_arquetipo}")
print(f"Filas en la tabla expandida:         {len(df_proc_arquetipos)}")
df_proc_arquetipos

Códigos únicos en el CSV:            168
Con al menos un arquetipo asociado:  80
Sin ningún arquetipo (N/A):          88
Filas en la tabla expandida:         187


,codigo_proceso,Código Arquetipo,Nombre Arquetipo
0,FID050210,N/A,N/A
1,FID080105,N/A,N/A
2,PAN040102,ARQ002,Abrir producto
3,PAN040103,ARQ002,Abrir producto
4,PAN040107,ARQ002,Abrir producto
...,...,...,...
182,T160668,N/A,N/A
183,T160669,ARQ002,Abrir producto
184,T160673,ARQ009,Autorizar tokenización
185,T160675,ARQ002,Abrir producto


## 4. Arquetipos × Capacidades de Proceso (tabla expandida)

Cruza los `Nombre Arquetipo` únicos del paso anterior contra el archivo `mapa_capacidades.csv`, evaluando si el nombre del arquetipo aparece **dentro** del campo `Arquetipos Relacionado` (búsqueda por substring, ya que el campo puede contener uno o varios arquetipos).

De cada fila coincidente se extraen:
- `Capacidad de Procesos (Nivel 3)` → renombrado como `capacidades`
- `Dominio`
- `Subdominio Nivel 1`
- `Subdominio Nivel 2`

El resultado `df_arq_capacidades` tiene **una fila por cada par (Nombre Arquetipo, capacidad)**.

In [5]:
import sys, os, importlib, pandas as pd
sys.path.insert(0, os.getcwd())

# ── Guard: reconstruye df_proc_arquetipos si la celda 3 no fue ejecutada ─────
if 'df_proc_arquetipos' not in vars() or 'arq_name_col' not in vars():
    import helper.match_arquetipos as _ma
    importlib.reload(_ma)
    from helper.match_arquetipos import load_arquetipos
    from pathlib import Path

    _df = pd.read_csv(Path('data/query_actividades_proc_q_actual.csv'), dtype=str)
    _, _df_arq_expl = load_arquetipos('data/arquetipos.csv')

    arq_code_col = next(c for c in _df_arq_expl.columns if 'Código' in c and 'Arquetipo' in c)
    arq_name_col = next(c for c in _df_arq_expl.columns if 'Nombre' in c and 'Arquetipo' in c)

    _df_codigos = (
        _df[['codigo_proceso']]
        .drop_duplicates()
        .assign(codigo_proceso=lambda x: x['codigo_proceso'].astype(str).str.strip())
        .reset_index(drop=True)
    )
    df_proc_arquetipos = (
        _df_codigos
        .merge(
            _df_arq_expl[['proc_code', arq_code_col, arq_name_col]].drop_duplicates(),
            left_on='codigo_proceso', right_on='proc_code', how='left',
        )
        [['codigo_proceso', arq_code_col, arq_name_col]]
        .fillna('N/A')
        .reset_index(drop=True)
    )
    print("(df_proc_arquetipos reconstruido desde archivos fuente)")

# ── Carga del mapa de capacidades ────────────────────────────────────────────
df_cap = pd.read_csv('data/mapa_capacidades.csv', dtype=str, encoding='latin-1')

_cap_cols = [
    'Capacidad de Procesos (Nivel 3)',
    'Dominio',
    'Subdominio Nivel 1',
    'Subdominio Nivel 2',
    'Arquetipos Relacionado',
]
df_cap_clean = (
    df_cap[_cap_cols]
    .dropna(subset=['Arquetipos Relacionado', 'Capacidad de Procesos (Nivel 3)'])
    .assign(**{'Arquetipos Relacionado': lambda x: x['Arquetipos Relacionado'].str.strip()})
    .reset_index(drop=True)
)

# ── Nombres de arquetipo únicos a buscar (excluye N/A) ───────────────────────
nombres_arq = [n for n in df_proc_arquetipos[arq_name_col].unique() if n != 'N/A']

# ── Cruce por substring: una fila por (arquetipo, capacidad) ─────────────────
fragmentos = []
for nombre in nombres_arq:
    mask = df_cap_clean['Arquetipos Relacionado'].str.contains(nombre, regex=False, na=False)
    coincidencias = df_cap_clean[mask].copy()
    coincidencias['Nombre Arquetipo'] = nombre
    fragmentos.append(coincidencias)

df_arq_capacidades = (
    pd.concat(fragmentos, ignore_index=True)
    .rename(columns={'Capacidad de Procesos (Nivel 3)': 'capacidades'})
    [['Nombre Arquetipo', 'capacidades', 'Dominio', 'Subdominio Nivel 1', 'Subdominio Nivel 2']]
    .drop_duplicates()
    .sort_values(['Nombre Arquetipo', 'capacidades'])
    .reset_index(drop=True)
)

print(f"Arquetipos únicos buscados:          {len(nombres_arq)}")
print(f"Con al menos una capacidad asociada: {df_arq_capacidades['Nombre Arquetipo'].nunique()}")
print(f"Filas en la tabla expandida:         {len(df_arq_capacidades)}")
df_arq_capacidades

Arquetipos únicos buscados:          17
Con al menos una capacidad asociada: 4
Filas en la tabla expandida:         170


,Nombre Arquetipo,capacidades,Dominio,Subdominio Nivel 1,Subdominio Nivel 2
0,Abrir producto,Ajustar parÃ¡metro,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...","Tarjeta crÃ©dito, tarjeta dÃ©bito, cheques, re..."
1,Abrir producto,Asignar sucursal de radicaciÃ³n,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...",NaN
2,Abrir producto,Asociar cuenta al producto,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...","DepÃ³sitos a la vista, depÃ³sitos a plazo, tar..."
3,Abrir producto,Asociar producto al cliente,Clientes,Relacionamiento con clientes,Por definir
4,Abrir producto,Asociar producto al cliente,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...",Por definir
...,...,...,...,...,...
165,Atender solicitudes de usuario,Validar identidad del usuario,Estrategia de los negocios,Modelo de Precio (pricing),No aplica
166,Atender solicitudes de usuario,Validar identidad del usuario,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...",Ciberseguridad
167,Atender solicitudes de usuario,Validar reglas acorde a la clasificaciÃ³n,Clientes,Estrategia de Clientes,Servicio
168,Atender solicitudes de usuario,Validar reglas acorde a la clasificaciÃ³n,Estrategia de los negocios,Modelo de Precio (pricing),No aplica
